In [2]:
import requests
import time
import numpy as np
from typing import Optional
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.optimize import minimize
import logging


# Setting up the API keys and trying a few submissions

In [14]:
API_KEY = "key_123"
BASE_URL = "http://localhost:8000/slacathon26"

headers = {
    "X-API-Key": API_KEY,
    "Content-Type": "application/json"
}

def run_validation(values, poll_interval=2):
    # Convert the 5-element list to the beamline task input dict
    # (q1, q2, q3, d2, d3) — this is what the active task expects
    input_dict = {
        "q1": values[0],
        "q2": values[1],
        "q3": values[2],
        "d2": values[3],
        "d3": values[4]
    }

    # Submit job using the current format: {"input": {...}}
    r = requests.post(f"{BASE_URL}/validate", headers=headers, json={"input": input_dict})
    r.raise_for_status()
    job = r.json()
    job_id = job["job_id"]
    print(f"Job started: {job_id} (status={job['status']})")

    # Poll until done
    while True:
        j = requests.get(f"{BASE_URL}/jobs/{job_id}", headers=headers).json()
        if j["status"] == "completed":
            return j.get("result")
        print(f"  ... still {j['status']}")
        time.sleep(poll_interval)


# Your two submissions
# Values are in order [q1, q2, q3, d2, d3] for the beamline task
result1 = run_validation([2.2547133301706257, -2.223405741870012, 0.9588998760031707, 0.033, 1.413])
print("First result:", result1)

result2 = run_validation([2.5537242710909087, -2.518264797355262, 1.0860652900429026, 0.033, 1.413])
print("Second result:", result2)

Job started: afee437f-da25-4d84-bbb0-5318e73cef4b (status=processing)
  ... still processing
First result: {'score': 1.5889397682278592, 'solved': False, 'message': 'Objective is 1.5889397682278592, expected minimal (less than 1e-4)', 'evaltime': 0.005319833755493164}
Job started: b5e0aab9-6c15-4620-992f-87ac94912086 (status=processing)
  ... still processing
Second result: {'score': 0.418888330879873, 'solved': False, 'message': 'Objective is 0.418888330879873, expected minimal (less than 1e-4)', 'evaltime': 0.0004324913024902344}


## Enable logger

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## Checking history of my submissions (10 last submissions)

In [17]:
response = requests.get(
    f"{BASE_URL}/history",
    headers={"X-API-Key": API_KEY}
)
if response.status_code == 200:
    data = response.json()
    logger.info(f"Total submissions: {data['total_submissions']}")
    logger.info(f"History: {data['history']}")

2026-07-07 18:59:42,383 - INFO - Total submissions: 0
2026-07-07 18:59:42,383 - INFO - History: []


# Gaussian process optimizer implementation (scikit learn)

In [3]:
class GPOptimizer:
    def __init__(self, api_key: str, base_url: str):
        """Initialize without fixed_values.
        It automatically fetches the active task's input schema via /task.
        """
        self.api_key = api_key
        self.base_url = base_url.rstrip("/")

        self.session = requests.Session()
        self.session.headers.update({
            "X-API-Key": api_key,
            "Content-Type": "application/json"
        })

        # Fetch current task info so we know the expected input format
        self.task_info = self._request("GET", "/task")
        self.input_labels = self.task_info.get("parameter_labels") or ["q1", "q2", "q3", "d2", "d3"]
        self.bounds = self.task_info.get("bounds") or [(-10.0, 10.0)] * len(self.input_labels)
        self.dim = len(self.input_labels)

        kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
        self.gp = GaussianProcessRegressor(
            kernel=kernel,
            n_restarts_optimizer=10,
            alpha=1e-6,
            normalize_y=True
        )

        self.X_observed: list[np.ndarray] = []
        self.y_observed: list[float] = []

        logger.info(f"GPOptimizer initialized for task '{self.task_info.get('name')}' "
                    f"({self.dim} parameters: {self.input_labels})")

    # ---------------- Internal helpers ----------------

    def _request(self, method: str, path: str, **kwargs) -> dict:
        url = f"{self.base_url}{path}"
        try:
            resp = self.session.request(method, url, timeout=kwargs.pop("timeout", 30), **kwargs)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            logger.error(f"{method} {path} failed: {e}")
            return {}

    def _wait_for_job(self, job_id: str, timeout: float = 300.0, poll_interval: float = 1.5) -> dict:
        """Poll until job is completed (or timeout). Returns the inner 'result' dict."""
        deadline = time.time() + timeout
        while time.time() < deadline:
            data = self._request("GET", f"/jobs/{job_id}")
            if data.get("status") == "completed":
                return data.get("result", {})
            if data.get("status") == "failed":
                logger.warning(f"Job {job_id} failed")
                return {}
            time.sleep(poll_interval)
        logger.warning(f"Job {job_id} timed out after {timeout}s")
        return {}

    # ---------------- Public API ----------------

    def query_objective(self, x: np.ndarray) -> tuple[float, dict]:
        """Submit using the task's input format (dict) and wait for result."""
        input_dict = {label: float(val) for label, val in zip(self.input_labels, x)}
        logger.info(f"Querying with input: {input_dict}")

        job = self._request("POST", "/validate", json={"input": input_dict})
        job_id = job.get("job_id")
        if not job_id:
            return float("inf"), {}

        logger.info(f"Validation job {job_id} started")
        result = self._wait_for_job(job_id)
        if not result:
            return float("inf"), {}

        score = result.get("score", float("inf"))
        logger.info(f"Job {job_id} → score={score:.6f}, solved={result.get('solved')}")
        return score, result

    def submit_to_leaderboard(self, x: np.ndarray) -> dict:
        input_dict = {label: float(val) for label, val in zip(self.input_labels, x)}
        result = self._request("POST", "/submit", json={"input": input_dict}, timeout=15)
        if result:
            logger.info(f"Submitted → rank {result.get('rank')}/{result.get('leaderboard_size')}")
        return result

    def view_leaderboard(self) -> dict:
        return self._request("GET", "/leaderboard")

    def get_history(self) -> Optional[dict]:
        return self._request("GET", "/history")

    # ---------------- GP / BO ----------------

    def acquisition_function(self, x: np.ndarray) -> float:
        if not self.X_observed:
            return 0.0
        try:
            mu, sigma = self.gp.predict(x.reshape(1, -1), return_std=True)
            return (mu - 2.0 * sigma)[0]  # LCB for minimization
        except Exception:
            return np.random.randn()

    def suggest_next_point(self, bounds: list[tuple] = None, n_restarts: int = 8) -> np.ndarray:
        if bounds is None:
            bounds = self.bounds
        if len(self.X_observed) < 3:
            return np.array([np.random.uniform(lo, hi) for lo, hi in bounds])

        best_x, best_val = None, float("inf")
        for _ in range(n_restarts):
            x0 = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])
            res = minimize(self.acquisition_function, x0, bounds=bounds, method="L-BFGS-B")
            if res.success and res.fun < best_val:
                best_val = res.fun
                best_x = res.x
        return best_x if best_x is not None else np.array([np.random.uniform(lo, hi) for lo, hi in bounds])

    def optimize(self,
                 bounds: list[tuple] = None,
                 n_iterations: int = 50,
                 target_score: float = 1e-3,
                 exploration_iterations: int = 10) -> tuple[Optional[np.ndarray], float, dict]:
        if bounds is None:
            bounds = self.bounds
        logger.info(f"Starting GP optimization (target < {target_score})")

        best_x: Optional[np.ndarray] = None
        best_score = float("inf")
        best_result: dict = {}

        for i in range(n_iterations):
            logger.info(f"Iter {i+1}/{n_iterations}")

            x = (np.array([np.random.uniform(lo, hi) for lo, hi in bounds])
                 if i < exploration_iterations else self.suggest_next_point(bounds))

            score, result = self.query_objective(x)
            if score == float("inf"):
                continue

            self.X_observed.append(x)
            self.y_observed.append(score)

            if score < best_score:
                best_score = score
                best_x = x
                best_result = result
                logger.info(f"New best: {best_score:.6f} @ {best_x}")

            if len(self.X_observed) >= 3:
                X = np.array(self.X_observed)
                y = np.array(self.y_observed)
                if np.all(np.isfinite(y)):
                    try:
                        self.gp.fit(X, y)
                    except Exception as e:
                        logger.warning(f"GP fit failed: {e}")

            if best_score < target_score:
                logger.info(f"Target reached: {best_score:.6f}")
                break

            time.sleep(1.0)

        logger.info(f"Done. Best score={best_score:.6f}, x={best_x}")
        if best_result:
            logger.info(f"Solved={best_result.get('solved')}, msg={best_result.get('message')}")
        return best_x, best_score, best_result

    def submit_best_to_leaderboard(self, x: np.ndarray) -> dict:
        """Convenience wrapper."""
        return self.submit_to_leaderboard(x)

In [6]:
# ====================== CONFIGURATION ======================
API_KEY = "key_123"
BASE_URL = "http://localhost:8000/slacathon26"

# Fixed values (last two parameters)
FIXED_VALUES = [1.0, 1.4]   # d2, d3

# Bounds for the first three variables
BOUNDS = [
    (1.0, 3.0),     # q1
    (-3.0, -2.0),   # q2
    (0.0, 2.0),     # q3
]
# ===========================================================

# Create optimizer (new signature — no fixed_values)
optimizer = GPOptimizer(
    api_key=API_KEY,
    base_url=BASE_URL
)

# Patch query_objective and submit_to_leaderboard so we can keep
# optimizing only the first 3 variables while the class works with
# the full task input (q1,q2,q3,d2,d3).
original_query = optimizer.query_objective
original_submit = optimizer.submit_to_leaderboard

def make_full(x):
    """Append the fixed values so the call uses the full 5-element input."""
    return list(x) + FIXED_VALUES

def patched_query(x):
    full_x = make_full(x)
    return original_query(np.array(full_x))

def patched_submit(x):
    full_x = make_full(x)
    return original_submit(np.array(full_x))

optimizer.query_objective = patched_query
optimizer.submit_to_leaderboard = patched_submit

# Run optimization using only the 3-variable bounds
best_x, best_score, best_result = optimizer.optimize(
    bounds=BOUNDS,
    n_iterations=250,
    target_score=1e-3,
    exploration_iterations=10
)

# ====================== FINAL REPORT ======================
print("\n" + "=" * 60)
print("OPTIMIZATION COMPLETE")
print("=" * 60)

if best_x is not None:
    full_solution = list(best_x) + FIXED_VALUES
    print(f"Best optimized variables (q1, q2, q3): {best_x}")
    print(f"Fixed variables (d2, d3):               {FIXED_VALUES}")
    print(f"Full solution [q1,q2,q3,d2,d3]:         {full_solution}")
    print(f"Best score:                             {best_score:.6f}")
    print(f"Target reached (< 1e-3):                {best_score < 1e-3}")

    if best_result:
        print(f"Solved:                                 {best_result.get('solved', False)}")
        print(f"Message:                                {best_result.get('message', 'N/A')}")

    # Automatically submit best result to leaderboard
    print("\n" + "-" * 60)
    print("Submitting best result to leaderboard...")
    submission = optimizer.submit_best_to_leaderboard(best_x)

    if submission:
        print(f"Submission successful!")
        print(f"Rank: {submission.get('rank')}/{submission.get('leaderboard_size')}")
    else:
        print("Submission failed or returned empty result.")
else:
    print("Optimization did not find a valid solution.")

print("=" * 60)

2026-07-07 20:25:09,053 - INFO - GPOptimizer initialized for task 'Beamline Guru' (5 parameters: ['q1', 'q2', 'q3', 'd2', 'd3'])
2026-07-07 20:25:09,054 - INFO - Starting GP optimization (target < 0.001)
2026-07-07 20:25:09,054 - INFO - Iter 1/250
2026-07-07 20:25:09,054 - INFO - Querying with input: {'q1': 2.8313448647121473, 'q2': -2.2525243547573144, 'q3': 0.04675611828759885, 'd2': 1.0, 'd3': 1.4}
2026-07-07 20:25:09,065 - INFO - Validation job 35af7d2e-16a5-49fd-a218-fb5debd02a3e started
2026-07-07 20:25:10,589 - INFO - Job 35af7d2e-16a5-49fd-a218-fb5debd02a3e → score=193.959604, solved=False
2026-07-07 20:25:10,590 - INFO - New best: 193.959604 @ [ 2.83134486 -2.25252435  0.04675612]
2026-07-07 20:25:11,591 - INFO - Iter 2/250
2026-07-07 20:25:11,594 - INFO - Querying with input: {'q1': 2.527486109967314, 'q2': -2.1721570525033904, 'q3': 1.570180586462811, 'd2': 1.0, 'd3': 1.4}
2026-07-07 20:25:11,617 - INFO - Validation job 44c3fa96-db1b-4613-9fa9-fe339c03c68a started
2026-07-07


OPTIMIZATION COMPLETE
Best optimized variables (q1, q2, q3): [ 1.75604497 -2.30618628  0.00911085]
Fixed variables (d2, d3):               [1.0, 1.4]
Full solution [q1,q2,q3,d2,d3]:         [np.float64(1.7560449665634383), np.float64(-2.3061862825715647), np.float64(0.009110848740854011), 1.0, 1.4]
Best score:                             52.156076
Target reached (< 1e-3):                False
Solved:                                 False
Message:                                Objective is 52.15607623235983, expected minimal (less than 1e-4)

------------------------------------------------------------
Submitting best result to leaderboard...
Submission successful!
Rank: 1/1


## Submit best result to the leaderboard

In [7]:
# Submit to leaderboard
print("\n" + "="*60)
print("SUBMITTING TO LEADERBOARD")
print("="*60)
submission_result = optimizer.submit_to_leaderboard(best_x)
if submission_result:
    print(f"Rank: {submission_result.get('rank')}/{submission_result.get('leaderboard_size')}")
    print(f"Score: {submission_result.get('score'):.6f}")

2026-07-07 20:37:20,520 - INFO - Submitted → rank None/1



SUBMITTING TO LEADERBOARD
Rank: None/1
Score: 52.156076


## View leaderboard

In [8]:
print("\n" + "="*60)
print("CURRENT LEADERBOARD")
print("="*60)
leaderboard = optimizer.view_leaderboard()
if leaderboard:
    for i, entry in enumerate(leaderboard.get('leaderboard', []), 1):
        solved_marker = "✓" if entry['solved'] else "✗"
        print(f"{i:2d}. [{solved_marker}] Score: {entry['score']:8.6f} | User: {entry['user']}")


CURRENT LEADERBOARD
 1. [✗] Score: 52.156076 | User: Alex
